# SILo single-session inspection and validation

This notebook performs **single-session QC of a GIAnT/SILo `experiment_summary_*.h5` output**.

The goal is to separate three questions:

1. **Structural validity** — are the expected datasets present and internally consistent?
2. **Algorithmic validity** — do source footprints, activity images, motion/discard masks, and traces behave sensibly?
3. **Biological plausibility** — are candidate sources localized to fluorescent dendritic structure rather than invalid-image boundaries or diffuse noise?

### Important interpretation rules

- High source SNR alone is **not** treated as evidence that a source is a real glutamatergic synapse.
- Plotted source traces are **not smoothed or filtered** by this notebook.
- `dF_denoised` and `events` are SILo-produced outputs; they are optional inspection layers, not independent validation.
- The stored `act_im` is the SILo detection/localization image available in the summary file. It should not be assumed to be a raw fluorescence image.
- Anatomy/activity correspondence and boundary enrichment are emphasized because weak/no-signal recordings can still produce apparently well-fit sources.

The notebook is written for the current GIAnT HDF5 schema:

`Path*/frame_info`, `Path*/global`, `Path*/sources`, and `Path*/visualizations`.


In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import distance_transform_edt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Point this at one SILo experiment-summary file.
SILO_FILE = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-11-06_803121\803121_2025-11-06_12-12-23\slap2\dynamic_data\source_extraction\experiment_summary.h5")
                 
# Display/QC parameters only. These do not modify the SILo output.
MEAN_IMAGE_Q = 0.975
ACT_IMAGE_Q = 0.995
EDGE_DISTANCE_PX = 2.5

# Temporal inspection. Traces are plotted directly from the stored arrays:
# no smoothing, filtering, or resampling is performed.
TRACE_WINDOW_S = 30.0
N_TRACE_SOURCES = 6
PLOT_STORED_DENOISED = False
PLOT_STORED_EVENTS = False

# Footprint/source gallery.
N_GALLERY_SOURCES = 6
CROP_RADIUS_PX = 10

# Optional CSV export at the bottom.
SAVE_QC_CSV = False

assert SILO_FILE.exists(), f"SILo file not found: {SILO_FILE.resolve()}"
print("SILo file:", SILO_FILE.resolve())


import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%matplotlib notebook

## 1. HDF5 readers and orientation helpers

In [ ]:
def _scalar(ds):
    return np.asarray(ds[()]).ravel()[0].item()


def is_row_major(f):
    return bool(_scalar(f["row_major"])) if "row_major" in f else False


def path_names(f):
    return sorted(
        [k for k in f.keys() if k.startswith("Path")],
        key=lambda x: int(x.replace("Path", "")),
    )


def read_object_string(ds):
    x = ds[()]

    if ds.dtype.kind != "O":
        arr = np.asarray(x)
        if arr.size == 1:
            value = arr.ravel()[0]
            return value.item() if hasattr(value, "item") else value
        return arr

    vals = []
    for ref in np.asarray(x).ravel():
        if isinstance(ref, h5py.Reference) and ref:
            arr = ds.file[ref][()]
            arr = np.asarray(arr)
            if arr.dtype.kind in "ui":
                vals.append("".join(map(chr, arr.ravel(order="F"))))
            else:
                vals.append(str(arr))
        else:
            vals.append(str(ref))
    return vals[0] if len(vals) == 1 else vals


def read_params(path):
    out = {}
    with h5py.File(path, "r") as f:
        if "params" not in f:
            return out
        for key, obj in f["params"].items():
            if isinstance(obj, h5py.Dataset):
                out[key] = read_object_string(obj)
    return out


def get_activity_channel_zero_based(f):
    # GIAnT stores activityChannel using MATLAB 1-based indexing.
    return int(round(_scalar(f["params/activityChannel"]))) - 1


def read_coords(f, path_name):
    # Returned columns are [z, row, col] in MATLAB 1-based image coordinates.
    a = np.asarray(f[f"{path_name}/sources/spatial/coords"][()], dtype=float)
    return a if is_row_major(f) else a.T


def source_rc_zero_based(f, path_name):
    coords = read_coords(f, path_name)
    if coords.size == 0:
        return np.empty((0, 2), dtype=float)
    return coords[:, 1:3] - 1.0


def read_peaks(f, path_name):
    a = np.asarray(f[f"{path_name}/visualizations/act_im_peaks"][()], dtype=float)
    return a if is_row_major(f) else a.T


def read_act_image(f, path_name, z=0):
    a = np.asarray(f[f"{path_name}/visualizations/act_im"][()], dtype=float)
    return a[z, :, :] if is_row_major(f) else a[:, :, z].T


def read_mean_image(f, path_name, channel=None, z=0):
    if channel is None:
        channel = get_activity_channel_zero_based(f)
    a = np.asarray(f[f"{path_name}/visualizations/mean_im"][()], dtype=float)
    return a[channel, z, :, :] if is_row_major(f) else a[:, :, z, channel].T


def read_profile(f, path_name, source, z=0):
    a = np.asarray(f[f"{path_name}/sources/spatial/profiles"][()], dtype=float)
    return a[source, z, :, :] if is_row_major(f) else a[:, :, z, source].T


def read_temporal(f, path_name, field, source, channel=None):
    if channel is None:
        channel = get_activity_channel_zero_based(f)
    a = np.asarray(f[f"{path_name}/sources/temporal/{field}"][()], dtype=float)
    return a[source, channel, :] if is_row_major(f) else a[:, channel, source]


def read_snr(f, path_name):
    return np.asarray(
        f[f"{path_name}/sources/temporal/SNR"][()],
        dtype=float,
    ).ravel()


def read_frame_vector(f, path_name, field):
    return np.asarray(f[f"{path_name}/frame_info/{field}"][()]).ravel()


def read_global_f(f, path_name):
    a = np.asarray(f[f"{path_name}/global/F"][()], dtype=float)
    return a.T if is_row_major(f) else a  # frames x channels


def n_sources(f, path_name):
    return read_coords(f, path_name).shape[0]


def source_profile_matrix(f, path_name):
    ns = n_sources(f, path_name)
    if ns == 0:
        shape = read_act_image(f, path_name).shape
        return np.empty((*shape, 0), dtype=float)
    return np.stack(
        [read_profile(f, path_name, s) for s in range(ns)],
        axis=-1,
    )


def finite_percentile(x, q):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if not finite.any():
        return np.nan
    return float(np.nanquantile(x[finite], q))


def normalize_overlay_image(im, q=0.99):
    im = np.asarray(im, dtype=float)
    finite = np.isfinite(im)
    if not finite.any():
        return np.zeros_like(im, dtype=float)

    vmax = finite_percentile(im, q)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(im[finite]))
    if not np.isfinite(vmax) or vmax <= 0:
        return np.zeros_like(im, dtype=float)

    out = im / vmax
    out = np.clip(out, 0, 1)
    out[~finite] = 0
    return out


def make_mean_activity_overlay(mean_im, act_im, q_mean=0.99, q_act=0.995):
    # Cyan = structural mean image; magenta = positive activity.
    mean_pos = np.clip(np.asarray(mean_im, float), 0, None)
    act_pos = np.clip(np.asarray(act_im, float), 0, None)

    mean_norm = normalize_overlay_image(mean_pos, q=q_mean)
    act_norm = normalize_overlay_image(act_pos, q=q_act)

    rgb = np.zeros((*mean_norm.shape, 3), dtype=float)
    rgb[..., 0] = act_norm
    rgb[..., 1] = mean_norm
    rgb[..., 2] = np.clip(mean_norm + act_norm, 0, 1)
    return np.clip(rgb, 0, 1)


def robust_imshow_limits(im, q=0.995, symmetric=False):
    im = np.asarray(im, float)
    finite = im[np.isfinite(im)]
    if finite.size == 0:
        return (0.0, 1.0)

    if symmetric:
        lim = float(np.quantile(np.abs(finite), q))
        if not np.isfinite(lim) or lim <= 0:
            lim = float(np.max(np.abs(finite)))
        return (-lim, lim)

    lo = float(np.quantile(finite, 1 - q))
    hi = float(np.quantile(finite, q))
    if not np.isfinite(lo):
        lo = float(np.min(finite))
    if not np.isfinite(hi):
        hi = float(np.max(finite))
    if hi <= lo:
        hi = lo + 1.0
    return lo, hi


with h5py.File(SILO_FILE, "r") as f:
    print("row_major:", is_row_major(f))
    print("paths:", path_names(f))
    print("activityChannel (MATLAB indexing):", get_activity_channel_zero_based(f) + 1)


## 2. Structural integrity

In [ ]:
REQUIRED = [
    "Z_depths",
    "frame_info/discard_frames",
    "frame_info/frame_line_idxs",
    "frame_info/offlineXshifts",
    "frame_info/offlineYshifts",
    "frame_info/onlineXshifts",
    "frame_info/onlineYshifts",
    "frame_info/onlineZshifts",
    "frame_info/trial_num_frames",
    "global/F",
    "sources/spatial/coords",
    "sources/spatial/profiles",
    "sources/temporal/F0",
    "sources/temporal/SNR",
    "sources/temporal/dF_denoised",
    "sources/temporal/dF_ls",
    "sources/temporal/events",
    "visualizations/act_im",
    "visualizations/act_im_peaks",
    "visualizations/mean_im",
]


def validate_file(path):
    rows = []

    with h5py.File(path, "r") as f:
        rows.append(("/", "row_major present", "row_major" in f))
        rows.append(("/", "params present", "params" in f))

        for pn in path_names(f):
            g = f[pn]

            for rel in REQUIRED:
                rows.append((pn, f"present: {rel}", rel in g))

            coords = read_coords(f, pn)
            peaks = read_peaks(f, pn)
            ns = coords.shape[0]

            trial_counts = read_frame_vector(f, pn, "trial_num_frames").astype(int)
            nf = int(trial_counts.sum())

            prof_ds = g["sources/spatial/profiles"]
            f0_ds = g["sources/temporal/F0"]
            snr = read_snr(f, pn)
            global_f = read_global_f(f, pn)
            discard = read_frame_vector(f, pn, "discard_frames")

            prof_ns = prof_ds.shape[0] if is_row_major(f) else prof_ds.shape[-1]
            f0_ns = f0_ds.shape[0] if is_row_major(f) else f0_ds.shape[-1]

            checks = {
                "coords are Nx3": coords.ndim == 2 and coords.shape[1] == 3,
                "peaks are Nx3": peaks.ndim == 2 and peaks.shape[1] == 3,
                "profiles source count matches coords": prof_ns == ns,
                "F0 source count matches coords": f0_ns == ns,
                "SNR source count matches coords": len(snr) == ns,
                "peak/source count match": peaks.shape[0] == ns,
                "global/F frame count matches trial counts": global_f.shape[0] == nf,
                "discard frame count matches trial counts": len(discard) == nf,
                "finite source coordinates": bool(np.isfinite(coords).all()) if ns else True,
                "finite SNR": bool(np.isfinite(snr).all()) if ns else True,
            }

            for name, ok in checks.items():
                rows.append((pn, name, bool(ok)))

    return pd.DataFrame(rows, columns=["scope", "check", "pass"])


validation = validate_file(SILO_FILE)

display(validation.groupby("scope")["pass"].agg(["count", "sum"]))

failed = validation.loc[~validation["pass"]]
if len(failed):
    display(failed)
else:
    print("All structural checks passed.")


A structural pass means the file is internally consistent enough to inspect. It does **not** establish that the detected sources are biological.

## 3. Parameter audit

In [ ]:
params = read_params(SILO_FILE)

param_table = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in sorted(params.items())]
)

display(param_table)

# Parameters most relevant to the current source-detection QC.
# "Working compromise" is intentionally not presented as a universal optimum.
working_compromise = {
    "peakth": 5,
    "nanThresh": 0.50,
    "VIF": 1.20,
    "motionThresh": 2.5,
    "minPeakDistance": 1,
    "sigma_px": 1.0,
    "tau_s": 0.02,
}

focus_rows = []
for key, target in working_compromise.items():
    focus_rows.append(
        {
            "parameter": key,
            "stored_value": params.get(key, np.nan),
            "current_working_compromise": target,
        }
    )

display(pd.DataFrame(focus_rows))


### Parameter interpretation

For the present QC strategy, the parameters that most directly affect false-positive localization are:

- **`peakth`**: native SILo activity-image peak threshold.
- **`nanThresh`**: how much missing support a pixel may have while remaining eligible.
- **`VIF`**: variance inflation used in the activity/noise model.
- **`motionThresh`**: frame censoring stringency.

The working-compromise column is included so repeated sessions can be audited against the same intended extraction regime. It is **not** a claim that those values are optimal for every recording.


## 4. Session/path summary

In [ ]:
summary_rows = []

with h5py.File(SILO_FILE, "r") as f:
    analyze_hz = float(_scalar(f["params/analyzeHz"])) if "params/analyzeHz" in f else np.nan

    for pn in path_names(f):
        discard = read_frame_vector(f, pn, "discard_frames").astype(bool)
        snr = read_snr(f, pn)
        ns = n_sources(f, pn)
        z_depth = float(np.asarray(f[f"{pn}/Z_depths"][()]).ravel()[0])

        summary_rows.append(
            {
                "path": pn,
                "z_depth": z_depth,
                "frames": len(discard),
                "duration_s": len(discard) / analyze_hz if np.isfinite(analyze_hz) and analyze_hz > 0 else np.nan,
                "sources": ns,
                "discard_frames": int(discard.sum()),
                "discard_pct": 100 * discard.mean() if discard.size else np.nan,
                "SNR_min": float(np.nanmin(snr)) if snr.size else np.nan,
                "SNR_median": float(np.nanmedian(snr)) if snr.size else np.nan,
                "SNR_max": float(np.nanmax(snr)) if snr.size else np.nan,
            }
        )

session_summary = pd.DataFrame(summary_rows)
display(session_summary)


## 5. Channel sanity check

In [ ]:
with h5py.File(SILO_FILE, "r") as f:
    pns = path_names(f)
    activity_ch = get_activity_channel_zero_based(f)

    for pn in pns:
        global_f = read_global_f(f, pn)
        n_channels = global_f.shape[1]

        fig, axes = plt.subplots(
            1,
            n_channels + 1,
            figsize=(5 * (n_channels + 1), 4),
            squeeze=False,
        )
        axes = axes[0]

        for ch in range(n_channels):
            mean_im = read_mean_image(f, pn, channel=ch)
            lo, hi = robust_imshow_limits(mean_im, q=0.99)
            axes[ch].imshow(mean_im, cmap="gray", vmin=lo, vmax=hi)
            suffix = "  <-- activityChannel" if ch == activity_ch else ""
            axes[ch].set_title(f"{pn}: mean channel {ch + 1}{suffix}")
            axes[ch].axis("off")

        ax = axes[-1]
        for ch in range(n_channels):
            # Stored global F, plotted directly; no temporal filtering.
            ax.plot(global_f[:, ch], lw=0.6, label=f"ch {ch + 1}")
        ax.set_title(f"{pn}: global F")
        ax.set_xlabel("sample")
        ax.set_ylabel("fluorescence")
        ax.legend(frameon=False)

        fig.tight_layout()
        plt.show()


Confirm here that the configured `activityChannel` is physically the detector/channel containing iGluSnFR. A perfectly functioning source detector operating on the wrong channel can still produce numerically plausible output.

## 6. Mean image, SILo activity image, overlay, and detected sources

In [ ]:
with h5py.File(SILO_FILE, "r") as f:
    for pn in path_names(f):
        mean_im = read_mean_image(f, pn)
        act_im = read_act_image(f, pn)
        rc = source_rc_zero_based(f, pn)

        overlay = make_mean_activity_overlay(
            mean_im,
            act_im,
            q_mean=MEAN_IMAGE_Q,
            q_act=ACT_IMAGE_Q,
        )

        fig, axes = plt.subplots(1, 4, figsize=(20, 5),sharex=True,sharey=True)

        lo, hi = robust_imshow_limits(mean_im, q=0.9)
        axes[0].imshow(mean_im, cmap="gray", vmin=lo, vmax=hi)
        axes[0].set_title(f"{pn}: mean image")
        axes[0].axis("off")

        vmin, vmax = robust_imshow_limits(act_im, q=ACT_IMAGE_Q, symmetric=True)
        axes[1].imshow(act_im, cmap="bwr", vmin=vmin, vmax=vmax)
        axes[1].set_facecolor("k")
        axes[1].set_title(f"{pn}: stored SILo activity image")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_facecolor("k")
        axes[2].set_title("cyan = mean; magenta = positive activity")
        axes[2].axis("off")

        axes[3].imshow(overlay)
        if len(rc):
            axes[3].scatter(
                rc[:, 1],
                rc[:, 0],
                s=28,
                facecolors="none",
                edgecolors="yellow",
                linewidths=0.9,
            )
            for i, (r, c) in enumerate(rc):
                axes[3].text(
                    c + 1.5,
                    r + 1.5,
                    str(i),
                    color="yellow",
                    fontsize=7,
                )
        axes[3].set_title(f"{pn}: detected sources (n={len(rc)})")
        axes[3].axis("off")

        fig.tight_layout()
        plt.show()


### What to look for

A convincing glutamate source should ideally satisfy several conditions simultaneously:

- localized positive activity rather than broad diffuse structure;
- spatial correspondence with fluorescent dendrite;
- not concentrated at the edge of the valid imaging support;
- a compact source footprint centered near the detected seed;
- a plausible unfiltered fluorescence transient.

The next sections quantify these points rather than relying only on visual inspection.


## 7. Invalid-support boundary enrichment

In [ ]:
def sample_image_at_sources(im, rc):
    vals = np.full(len(rc), np.nan, dtype=float)
    nr, nc = im.shape
    for i, (r, c) in enumerate(rc):
        rr = int(round(r))
        cc = int(round(c))
        if 0 <= rr < nr and 0 <= cc < nc:
            vals[i] = im[rr, cc]
    return vals


def source_intensity_percentiles(mean_im, rc):
    finite_vals = np.asarray(mean_im, float)
    finite_vals = finite_vals[np.isfinite(finite_vals)]

    if finite_vals.size == 0:
        return np.full(len(rc), np.nan)

    src_vals = sample_image_at_sources(mean_im, rc)
    out = np.full(len(src_vals), np.nan)

    sorted_vals = np.sort(finite_vals)
    for i, value in enumerate(src_vals):
        if np.isfinite(value):
            out[i] = 100.0 * np.searchsorted(sorted_vals, value, side="right") / len(sorted_vals)
    return out


boundary_rows = []

with h5py.File(SILO_FILE, "r") as f:
    for pn in path_names(f):
        act_im = read_act_image(f, pn)
        mean_im = read_mean_image(f, pn)
        rc = source_rc_zero_based(f, pn)

        valid = np.isfinite(act_im)
        dist = distance_transform_edt(valid)

        src_dist = sample_image_at_sources(dist, rc)
        src_int_pct = source_intensity_percentiles(mean_im, rc)

        valid_dist = dist[valid]
        valid_edge_frac = (
            float(np.mean(valid_dist <= EDGE_DISTANCE_PX))
            if valid_dist.size
            else np.nan
        )
        source_edge_frac = (
            float(np.mean(src_dist <= EDGE_DISTANCE_PX))
            if src_dist.size
            else np.nan
        )

        enrichment = (
            source_edge_frac / valid_edge_frac
            if np.isfinite(source_edge_frac)
            and np.isfinite(valid_edge_frac)
            and valid_edge_frac > 0
            else np.nan
        )

        abs_act = np.abs(act_im)
        edge_values = abs_act[valid & (dist <= EDGE_DISTANCE_PX)]
        interior_values = abs_act[valid & (dist >= 4.0)]

        boundary_rows.append(
            {
                "path": pn,
                "sources": len(rc),
                "median_source_distance_to_boundary_px": float(np.nanmedian(src_dist)) if len(src_dist) else np.nan,
                f"source_fraction_within_{EDGE_DISTANCE_PX:g}px": source_edge_frac,
                f"valid_pixel_fraction_within_{EDGE_DISTANCE_PX:g}px": valid_edge_frac,
                "boundary_enrichment_ratio": enrichment,
                "median_source_mean_image_percentile": float(np.nanmedian(src_int_pct)) if len(src_int_pct) else np.nan,
                "median_abs_act_edge": float(np.nanmedian(edge_values)) if edge_values.size else np.nan,
                "median_abs_act_interior": float(np.nanmedian(interior_values)) if interior_values.size else np.nan,
                "edge_to_interior_abs_act_ratio": (
                    float(np.nanmedian(edge_values) / np.nanmedian(interior_values))
                    if edge_values.size
                    and interior_values.size
                    and np.nanmedian(interior_values) > 0
                    else np.nan
                ),
            }
        )

        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

        im = axes[0].imshow(dist, cmap="viridis")
        if len(rc):
            axes[0].scatter(
                rc[:, 1],
                rc[:, 0],
                s=35,
                facecolors="none",
                edgecolors="red",
            )
        axes[0].set_title(f"{pn}: distance to invalid boundary")
        axes[0].axis("off")
        plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="pixels")

        if valid_dist.size:
            axes[1].hist(
                valid_dist,
                bins=np.arange(0.5, np.nanmax(valid_dist) + 1.5, 1),
                density=True,
                alpha=0.55,
                label="all valid pixels",
            )
        if len(src_dist):
            axes[1].hist(
                src_dist[np.isfinite(src_dist)],
                bins=np.arange(0.5, max(2.5, np.nanmax(valid_dist)) + 1.5, 1),
                density=True,
                histtype="step",
                linewidth=2,
                label="sources",
            )
        axes[1].axvline(EDGE_DISTANCE_PX, ls="--", lw=1)
        axes[1].set_xlabel("distance to invalid boundary (px)")
        axes[1].set_ylabel("density")
        axes[1].set_title("source vs available-pixel geometry")
        axes[1].legend(frameon=False)

        if len(src_dist):
            axes[2].scatter(src_dist, src_int_pct, s=30)
            for i, (x, y) in enumerate(zip(src_dist, src_int_pct)):
                if np.isfinite(x) and np.isfinite(y):
                    axes[2].text(x, y, str(i), fontsize=7)
        axes[2].axvline(EDGE_DISTANCE_PX, ls="--", lw=1)
        axes[2].set_xlabel("source distance to boundary (px)")
        axes[2].set_ylabel("mean-image intensity percentile")
        axes[2].set_ylim(0, 100)
        axes[2].set_title("anatomy vs boundary location")

        fig.tight_layout()
        plt.show()


boundary_qc = pd.DataFrame(boundary_rows)
display(boundary_qc)


### Interpreting boundary QC

`boundary_enrichment_ratio` compares the fraction of detected sources near the invalid-data edge with the fraction expected from the geometry of all available pixels.

- **~1**: sources are no more edge-associated than expected from available area.
- **>1**: sources are enriched near invalid-support boundaries.
- A large `edge_to_interior_abs_act_ratio` suggests that the stored activity statistic itself is stronger near these boundaries.

These are artifact diagnostics, not formal significance tests.


## 8. Footprint numerical QC

In [ ]:
footprint_rows = []

with h5py.File(SILO_FILE, "r") as f:
    for pn in path_names(f):
        profiles = source_profile_matrix(f, pn)
        rc = source_rc_zero_based(f, pn)
        snr = read_snr(f, pn)

        for s in range(profiles.shape[-1]):
            fp = profiles[..., s]
            finite = np.isfinite(fp)
            positive = finite & (fp > 0)

            fp_sum = float(np.nansum(fp))
            neg_pixels = int(np.sum(finite & (fp < -1e-8)))
            support_px = int(np.sum(positive))

            # Weighted footprint centroid in Python 0-based display coordinates.
            w = np.where(finite, np.clip(fp, 0, None), 0.0)
            wsum = float(np.sum(w))

            if wsum > 0:
                yy, xx = np.indices(fp.shape)
                cy = float(np.sum(yy * w) / wsum)
                cx = float(np.sum(xx * w) / wsum)
                seed_dist = float(np.hypot(cy - rc[s, 0], cx - rc[s, 1]))
            else:
                cy = cx = seed_dist = np.nan

            footprint_rows.append(
                {
                    "path": pn,
                    "source": s,
                    "SNR": snr[s] if s < len(snr) else np.nan,
                    "profile_sum": fp_sum,
                    "profile_sum_error": abs(fp_sum - 1.0),
                    "negative_pixels_lt_-1e-8": neg_pixels,
                    "positive_support_px": support_px,
                    "profile_centroid_to_seed_px": seed_dist,
                }
            )

footprint_qc = pd.DataFrame(footprint_rows)
display(footprint_qc)

if len(footprint_qc):
    display(
        footprint_qc.groupby("path").agg(
            sources=("source", "count"),
            median_SNR=("SNR", "median"),
            max_profile_sum_error=("profile_sum_error", "max"),
            negative_pixels=("negative_pixels_lt_-1e-8", "sum"),
            median_support_px=("positive_support_px", "median"),
            median_centroid_to_seed_px=("profile_centroid_to_seed_px", "median"),
        )
    )


## 9. Source gallery: anatomy, activity, and footprint

In [ ]:
def representative_source_indices(snr, n):
    snr = np.asarray(snr, float)
    if snr.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(snr)
    n = min(int(n), len(order))
    if n == len(order):
        return order

    # Sample across the SNR ranking rather than showing only the strongest sources.
    rank_positions = np.linspace(0, len(order) - 1, n).round().astype(int)
    return order[rank_positions]


def crop_bounds(shape, r, c, radius):
    rr = int(round(r))
    cc = int(round(c))
    r0 = max(0, rr - radius)
    r1 = min(shape[0], rr + radius + 1)
    c0 = max(0, cc - radius)
    c1 = min(shape[1], cc + radius + 1)
    return r0, r1, c0, c1


with h5py.File(SILO_FILE, "r") as f:
    for pn in path_names(f):
        mean_im = read_mean_image(f, pn)
        act_im = read_act_image(f, pn)
        rc = source_rc_zero_based(f, pn)
        snr = read_snr(f, pn)

        inds = representative_source_indices(snr, N_GALLERY_SOURCES)
        if len(inds) == 0:
            print(f"{pn}: no detected sources")
            continue

        fig, axes = plt.subplots(
            len(inds),
            3,
            figsize=(10, 3.2 * len(inds)),
            squeeze=False,
        )

        for row, s in enumerate(inds):
            r, c = rc[s]
            r0, r1, c0, c1 = crop_bounds(mean_im.shape, r, c, CROP_RADIUS_PX)

            mean_crop = mean_im[r0:r1, c0:c1]
            act_crop = act_im[r0:r1, c0:c1]
            fp_crop = read_profile(f, pn, int(s))[r0:r1, c0:c1]

            lo, hi = robust_imshow_limits(mean_crop, q=0.99)
            axes[row, 0].imshow(mean_crop, cmap="gray", vmin=lo, vmax=hi)
            axes[row, 0].scatter(
                c - c0,
                r - r0,
                s=50,
                facecolors="none",
                edgecolors="yellow",
            )
            axes[row, 0].set_title(f"S{s} mean; SNR={snr[s]:.2f}")
            axes[row, 0].axis("off")

            vmin, vmax = robust_imshow_limits(act_crop, q=ACT_IMAGE_Q, symmetric=True)
            axes[row, 1].imshow(act_crop, cmap="bwr", vmin=vmin, vmax=vmax)
            axes[row, 1].scatter(
                c - c0,
                r - r0,
                s=50,
                facecolors="none",
                edgecolors="yellow",
            )
            axes[row, 1].set_title("stored activity")
            axes[row, 1].axis("off")

            mean_norm = normalize_overlay_image(np.clip(mean_crop, 0, None), q=0.99)
            fp_norm = normalize_overlay_image(np.clip(fp_crop, 0, None), q=0.999)

            rgb = np.zeros((*mean_norm.shape, 3), dtype=float)
            rgb[..., 0] = fp_norm
            rgb[..., 1] = mean_norm
            rgb[..., 2] = np.clip(mean_norm + fp_norm, 0, 1)

            axes[row, 2].imshow(np.clip(rgb, 0, 1))
            axes[row, 2].set_title("cyan = mean; magenta = footprint")
            axes[row, 2].axis("off")

        fig.suptitle(f"{pn}: representative sources across SNR rank", y=0.99,fontsize=18)
        fig.tight_layout()
        plt.show()


## 10. Unfiltered source-trace inspection

In [ ]:
with h5py.File(SILO_FILE, "r") as f:
    analyze_hz = float(_scalar(f["params/analyzeHz"])) if "params/analyzeHz" in f else np.nan

    if not np.isfinite(analyze_hz) or analyze_hz <= 0:
        raise ValueError("A positive params/analyzeHz is required for temporal QC.")

    n_show = max(1, int(round(TRACE_WINDOW_S * analyze_hz)))

    for pn in path_names(f):
        snr = read_snr(f, pn)
        inds = representative_source_indices(snr, N_TRACE_SOURCES)

        if len(inds) == 0:
            print(f"{pn}: no detected sources")
            continue

        fig, axes = plt.subplots(
            len(inds),
            1,
            figsize=(14, 2.2 * len(inds)),
            sharex=True,
            squeeze=False,
        )
        axes = axes[:, 0]

        for ax, s in zip(axes, inds):
            dF = read_temporal(f, pn, "dF_ls", int(s))
            n = min(len(dF), n_show)
            t = np.arange(n, dtype=float) / analyze_hz

            # Directly plot stored dF_ls: no smoothing/filtering.
            ax.plot(t, dF[:n], lw=0.7, label="stored dF_ls")

            if PLOT_STORED_DENOISED:
                den = read_temporal(f, pn, "dF_denoised", int(s))
                ax.plot(
                    t,
                    den[:n],
                    lw=0.8,
                    label="stored dF_denoised",
                )

            if PLOT_STORED_EVENTS:
                ev = read_temporal(f, pn, "events", int(s))
                finite_ev = ev[:n][np.isfinite(ev[:n])]
                if finite_ev.size and np.nanmax(np.abs(finite_ev)) > 0:
                    scale = np.nanmax(np.abs(dF[:n])) / np.nanmax(np.abs(finite_ev))
                    ax.plot(
                        t,
                        ev[:n] * scale,
                        lw=0.6,
                        alpha=0.8,
                        label="stored events (display-scaled)",
                    )

            ax.set_ylabel(f"S{s}\nΔF")
            ax.set_title(f"{pn} source {s} | SNR={snr[s]:.2f}", loc="left", fontsize=9)
            ax.spines[["top", "right"]].set_visible(False)

        axes[-1].set_xlabel("time (s)")
        axes[0].legend(frameon=False, loc="upper right")
        fig.suptitle(
            f"{pn}: first {TRACE_WINDOW_S:g} s of stored source traces",
            y=0.97,fontsize=18,
        )
        fig.tight_layout()
        plt.show()


For weak-expression recordings, specifically look for whether the unfiltered `dF_ls` traces contain visually credible fast positive transients or are dominated by noise-like fluctuations. The optional stored denoised/event traces can help understand what SILo inferred, but they should not be treated as an independent ground truth.

## 11. Motion/discard-frame QC

In [ ]:
motion_rows = []
motion_corr_rows = []

with h5py.File(SILO_FILE, "r") as f:
    analyze_hz = float(_scalar(f["params/analyzeHz"])) if "params/analyzeHz" in f else np.nan

    for pn in path_names(f):
        ox = read_frame_vector(f, pn, "offlineXshifts").astype(float)
        oy = read_frame_vector(f, pn, "offlineYshifts").astype(float)
        discard = read_frame_vector(f, pn, "discard_frames").astype(bool)

        n = min(len(ox), len(oy), len(discard))
        ox, oy, discard = ox[:n], oy[:n], discard[:n]

        motion_mag = np.hypot(ox, oy)
        t = np.arange(n) / analyze_hz if np.isfinite(analyze_hz) and analyze_hz > 0 else np.arange(n)

        motion_rows.append(
            {
                "path": pn,
                "offline_x_std_px": float(np.nanstd(ox)),
                "offline_y_std_px": float(np.nanstd(oy)),
                "offline_motion_95pct_px": float(np.nanquantile(motion_mag, 0.95)),
                "offline_motion_max_px": float(np.nanmax(motion_mag)),
                "discard_pct": 100 * discard.mean(),
            }
        )

        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        axes[0].plot(t, ox, lw=0.6, label="offline X")
        axes[0].plot(t, oy, lw=0.6, label="offline Y")
        axes[0].set_ylabel("shift (px)")
        axes[0].legend(frameon=False)
        axes[0].set_title(f"{pn}: offline registration residuals")

        axes[1].plot(t, motion_mag, lw=0.6, label="offline |XY|")
        if discard.any():
            axes[1].scatter(
                t[discard],
                motion_mag[discard],
                s=8,
                label="discarded",
            )
        axes[1].set_ylabel("|shift| (px)")
        axes[1].set_xlabel("time (s)" if np.isfinite(analyze_hz) and analyze_hz > 0 else "sample")
        axes[1].legend(frameon=False)

        fig.tight_layout()
        plt.show()

        # Correlation between instantaneous residual-motion magnitude and absolute
        # stored dF_ls. This uses the unfiltered stored vectors directly.
        for s in range(n_sources(f, pn)):
            dF = read_temporal(f, pn, "dF_ls", s)
            nn = min(len(dF), len(motion_mag))
            x = motion_mag[:nn]
            y = np.abs(dF[:nn])

            finite = np.isfinite(x) & np.isfinite(y)
            if finite.sum() >= 10 and np.nanstd(x[finite]) > 0 and np.nanstd(y[finite]) > 0:
                r = float(np.corrcoef(x[finite], y[finite])[0, 1])
            else:
                r = np.nan

            motion_corr_rows.append(
                {
                    "path": pn,
                    "source": s,
                    "corr_abs_dF_vs_offline_motion": r,
                }
            )

motion_qc = pd.DataFrame(motion_rows)
motion_source_corr = pd.DataFrame(motion_corr_rows)

display(motion_qc)

if len(motion_source_corr):
    display(
        motion_source_corr.groupby("path")["corr_abs_dF_vs_offline_motion"].agg(
            ["count", "median", "min", "max"]
        )
    )

    for pn, dfp in motion_source_corr.groupby("path"):
        fig, ax = plt.subplots(figsize=(7, 4))
        vals = dfp["corr_abs_dF_vs_offline_motion"].dropna()
        ax.hist(vals, bins=15)
        ax.axvline(0, lw=1)
        ax.set_xlabel("corr(|stored dF_ls|, offline motion magnitude)")
        ax.set_ylabel("sources")
        ax.set_title(f"{pn}: source/motion coupling")
        fig.tight_layout()
        plt.show()


## 12. Consolidated QC table

In [ ]:
qc_summary = (
    session_summary
    .merge(boundary_qc, on=["path", "sources"], how="left")
    .merge(motion_qc, on=["path", "discard_pct"], how="left")
)

if len(footprint_qc):
    fp_summary = (
        footprint_qc.groupby("path")
        .agg(
            max_profile_sum_error=("profile_sum_error", "max"),
            total_negative_profile_pixels=("negative_pixels_lt_-1e-8", "sum"),
            median_profile_support_px=("positive_support_px", "median"),
            median_profile_centroid_to_seed_px=("profile_centroid_to_seed_px", "median"),
        )
        .reset_index()
    )
    qc_summary = qc_summary.merge(fp_summary, on="path", how="left")

if len(motion_source_corr):
    mc_summary = (
        motion_source_corr.groupby("path")
        .agg(
            median_abs_motion_corr=(
                "corr_abs_dF_vs_offline_motion",
                lambda x: float(np.nanmedian(np.abs(x))),
            )
        )
        .reset_index()
    )
    qc_summary = qc_summary.merge(mc_summary, on="path", how="left")


def qc_notes(row):
    notes = []

    enrich = row.get("boundary_enrichment_ratio", np.nan)
    if np.isfinite(enrich) and enrich > 1.5:
        notes.append("sources enriched near invalid-data boundary")

    edge_ratio = row.get("edge_to_interior_abs_act_ratio", np.nan)
    if np.isfinite(edge_ratio) and edge_ratio > 2:
        notes.append("activity magnitude elevated near boundary")

    struct_pct = row.get("median_source_mean_image_percentile", np.nan)
    if np.isfinite(struct_pct) and struct_pct < 25:
        notes.append("sources preferentially occupy dim structural pixels")

    discard_pct = row.get("discard_pct", np.nan)
    if np.isfinite(discard_pct) and discard_pct > 1:
        notes.append("elevated discard-frame fraction")

    motion_corr = row.get("median_abs_motion_corr", np.nan)
    if np.isfinite(motion_corr) and motion_corr > 0.25:
        notes.append("source activity shows notable residual-motion coupling")

    if not notes:
        notes.append("no major automatic flag; inspect images/traces manually")

    return "; ".join(notes)


qc_summary["QC_notes"] = qc_summary.apply(qc_notes, axis=1)
display(qc_summary)


## 13. Manual biological validation checklist

Use the quantitative table together with the figures above.

### Stronger evidence for a credible SILo source
- positive activity hotspot overlaps a fluorescent dendritic segment;
- hotspot is compact rather than diffuse;
- seed is not simply sitting at an invalid-support boundary;
- footprint is localized and centered near the source seed;
- unfiltered `dF_ls` contains plausible fast positive transients;
- signal is not tightly coupled to residual registration motion;
- the source remains plausible when viewed independently of its reported SNR.

### Warning signs
- most hotspots occur at valid/invalid image boundaries;
- activity-map contrast is much larger at edges than in the interior;
- sources lie on dim/non-dendritic pixels;
- footprint is numerically valid but anatomically implausible;
- stored traces look noise-like despite apparently respectable SNR;
- detections disappear with only a modest increase in `peakth` or tightening of `nanThresh`.

### Final judgment
A session can be **numerically valid but biologically uninformative**. In a weak-expression or expression-check animal, a reasonable outcome is that SILo correctly processes the data structure while source detection is operating near the false-positive floor because true iGluSnFR transients are absent or too weak.


## 14. Optional QC export

In [ ]:
if SAVE_QC_CSV:
    csv_path = SILO_FILE.with_name(f"{SILO_FILE.stem}_qc_summary.csv")
    qc_summary.to_csv(csv_path, index=False)
    print("Saved:", csv_path.resolve())
else:
    print("SAVE_QC_CSV=False; no files written.")
